# 01 — Ingestion Pipeline
### Loading data into the three knowledge sources our RAG agent will query

This notebook builds the **data layer** for a multi-source RAG agent. We have three
kinds of knowledge, so we prepare three kinds of storage:

| Source | What it holds | Storage | Access pattern |
|---|---|---|---|
| Company Policies | HR/compliance PDFs | Pinecone vector index `company-policies` | semantic (vector) search |
| Product Manuals | spec/how-to PDFs | Pinecone vector index `product-manuals` | semantic (vector) search |
| Structured records | orders, inventory, customers | SQLite `company_data.db` | SQL queries |

**Run this notebook once** (or whenever your source documents change) to populate the
data. 

The second notebook, `02_rag_agent_chatbot.ipynb`, only *reads* what we create here — it
never re-ingests anything.

**What you'll do in order:**
1. Install packages & set API keys
2. Create two Pinecone indexes (one per document domain)
3. Chunk PDFs and embed them into Pinecone
4. Create a small SQLite database with sample structured data
5. Sanity-check every source with a test query


## 1. Install packages

Run once per environment.

In [ ]:
# %pip install -q -U langchain langchain-community langchain-openai \
#     langchain-text-splitters langchain-pinecone  pypdf


In [18]:
import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community", "langchain-openai", 
            "langchain-text-splitters", "langchain-pinecone",
            "pypdf"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


# langchain version: 1.3.6
# langchain-community version: 0.4.2
# langchain-openai version: 1.2.2
# langchain-text-splitters version: 1.1.2
# langchain-pinecone version: 0.2.13
# pypdf version: 6.10.2

langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-openai version: 1.2.2
langchain-text-splitters version: 1.1.2
langchain-pinecone version: 0.2.13
pypdf version: 6.10.2


## 2. Configuration & clients

In [2]:
from dotenv import load_dotenv
import os

# Load the .env file
load_dotenv()

True

In [3]:
import os

# --- Uncomment and fill in for a live demo; otherwise these must already be set ---
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["PINECONE_API_KEY"] = "pcsk-..."

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]

POLICY_INDEX_NAME = "company-policies-index"
PRODUCT_INDEX_NAME = "product-manuals-index"
SQLITE_PATH = "company_data.db"          # created in this notebook's working directory

EMBEDDING_MODEL = "text-embedding-3-small"   # 1536 dimensions
EMBEDDING_DIM = 1536


In [5]:
# Test keys
import os
from openai import OpenAI
from pinecone import Pinecone

# 1. Ensure keys are pulled from environment variables
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")

print("--- Testing API Keys ---")

# 2. Test OpenAI Key
try:
    # Initialize the client (automatically uses os.environ["OPENAI_API_KEY"])
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # Fast and cost-efficient model
        messages=[
            {"role": "user", "content": "Hi"}
        ]
    )
    print("Response from OpenAI:", response.choices[0].message.content)
    print("✅ OpenAI API Key: SUCCESS (Valid connection)")
    
except Exception as e:
    print(f"❌ OpenAI API Key: FAILED. Error: {e}")

# 3. Test Pinecone Key
try:
    # Initialize the client (automatically uses os.environ["PINECONE_API_KEY"])
    pc = Pinecone()
    pc.list_indexes() # Make a minimal API call to list existing vector indexes
    print("✅ Pinecone API Key: SUCCESS (Valid connection)")
    
except Exception as e:
    print(f"❌ Pinecone API Key: FAILED. Error: {e}")


--- Testing API Keys ---
Response from OpenAI: Hello! How can I assist you today?
✅ OpenAI API Key: SUCCESS (Valid connection)
✅ Pinecone API Key: SUCCESS (Valid connection)


In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, api_key=OPENAI_API_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)

print("Pinecone client ready. Existing indexes:", [i["name"] for i in pc.list_indexes()])


Pinecone client ready. Existing indexes: []


## 3. Create the Pinecone indexes

- One index per knowledge *domain*.
- Keeping policies and product manuals in **separate
indexes** (rather than one index with a metadata filter) makes the router's job simpler
later: "policy" and "product" map 1:1 to an index, and it's easy to add a 4th domain
without touching the other two.

In [7]:
def create_index_if_missing(index_name: str, dimension: int = EMBEDDING_DIM):
    existing = [i["name"] for i in pc.list_indexes()]
    if index_name in existing:
        print(f"'{index_name}' already exists — skipping.")
        return
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Created index '{index_name}'.")

create_index_if_missing(POLICY_INDEX_NAME)
create_index_if_missing(PRODUCT_INDEX_NAME)


Created index 'company-policies-index'.
Created index 'product-manuals-index'.


## 4. Chunk & embed PDFs into Pinecone

`ingest_pdf` is generic: point it at any PDF and the index it belongs in. In a real
project you'd loop this over a folder of policy PDFs and a folder of manual PDFs.

For this demo, put a couple of sample PDFs in `./sample_docs/policies/` and
`./sample_docs/manuals/` (or point `pdf_path` at your own files).

In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def ingest_pdf(pdf_path: str, index_name: str, chunk_size: int = 1000, chunk_overlap: int = 150):
    """Load a PDF, split it into overlapping chunks, embed, and upsert to Pinecone."""
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(pages)

    # tag each chunk with its origin file for traceability in citations
    for c in chunks:
        c.metadata["source_file"] = os.path.basename(pdf_path)

    store = PineconeVectorStore(index=pc.Index(index_name), embedding=embeddings)
    store.add_documents(chunks)
    print(f"Ingested {len(chunks)} chunks from '{pdf_path}' -> index '{index_name}'")


C:\Users\hi\AppData\Local\Temp\ipykernel_16392\2537291163.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [9]:
import glob

# Ingest every PDF in sample_docs/policies/ into the policy index
for path in glob.glob("sample_docs/policies/*.pdf"):
    ingest_pdf(path, POLICY_INDEX_NAME)

# Ingest every PDF in sample_docs/manuals/ into the product index
for path in glob.glob("sample_docs/manuals/*.pdf"):
    ingest_pdf(path, PRODUCT_INDEX_NAME)


Ingested 8 chunks from 'sample_docs/policies\Loxford_Company_Policy.pdf' -> index 'company-policies-index'
Ingested 7 chunks from 'sample_docs/manuals\Loxford_X200_Manual.pdf' -> index 'product-manuals-index'


> **No PDFs handy?** Run the cell below to ingest a couple of plain-text snippets
> directly (skips the PDF loader) so you can still test the rest of the pipeline.

In [ ]:
# from langchain_core.documents import Document

# demo_policy_docs = [
#     Document(page_content="Employees may work remotely up to 3 days per week with manager approval. "
#                            "Fully remote arrangements require VP sign-off and a documented home-office setup.",
#              metadata={"source_file": "remote_work_policy.txt"}),
#     Document(page_content="Products may be returned within 30 days of purchase for a full refund, "
#                            "provided the original packaging and receipt are included.",
#              metadata={"source_file": "returns_policy.txt"}),
# ]

# demo_product_docs = [
#     Document(page_content="X200 Router — Factory reset: hold the reset button for 10 seconds until the "
#                            "LED blinks amber. This restores factory defaults and erases saved Wi-Fi credentials.",
#              metadata={"source_file": "x200_manual.txt"}),
#     Document(page_content="X200 Router — Warranty: 2 years from date of purchase, covering manufacturing "
#                            "defects. Does not cover water damage or unauthorized modification.",
#              metadata={"source_file": "x200_manual.txt"}),
# ]

# PineconeVectorStore(index=pc.Index(POLICY_INDEX_NAME), embedding=embeddings).add_documents(demo_policy_docs)
# PineconeVectorStore(index=pc.Index(PRODUCT_INDEX_NAME), embedding=embeddings).add_documents(demo_product_docs)
# print("Demo documents ingested.")


## 5. Build the SQL data source

The third knowledge source is structured, not document-shaped — think inventory
counts, order records, customer rows. That data belongs in a real database, not a
vector index, because we want **exact** filtering and aggregation ("how many do we
have left"), not semantic similarity.

In [10]:
import sqlite3

conn = sqlite3.connect(SQLITE_PATH)
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS inventory;
DROP TABLE IF EXISTS orders;

CREATE TABLE inventory (
    product_id   TEXT PRIMARY KEY,
    product_name TEXT NOT NULL,
    quantity     INTEGER NOT NULL,
    unit_price   REAL NOT NULL
);

CREATE TABLE orders (
    order_id     INTEGER PRIMARY KEY,
    product_id   TEXT NOT NULL,
    customer     TEXT NOT NULL,
    quantity     INTEGER NOT NULL,
    order_date   TEXT NOT NULL,
    FOREIGN KEY (product_id) REFERENCES inventory(product_id)
);
""")

cur.executemany(
    "INSERT INTO inventory (product_id, product_name, quantity, unit_price) VALUES (?, ?, ?, ?)",
    [
        ("X200", "X200 Router", 42, 89.99),
        ("Y150", "Y150 Switch", 130, 45.50),
        ("Z900", "Z900 Access Point", 17, 129.00),
    ],
)

cur.executemany(
    "INSERT INTO orders (order_id, product_id, customer, quantity, order_date) VALUES (?, ?, ?, ?, ?)",
    [
        (1, "X200", "Acme Corp", 5, "2026-05-01"),
        (2, "Y150", "Globex Inc", 20, "2026-05-03"),
        (3, "X200", "Initech", 2, "2026-06-10"),
    ],
)

conn.commit()
conn.close()
print(f"SQLite database created at ./{SQLITE_PATH}")

SQLite database created at ./company_data.db


## 6. Sanity-check every source

Before handing off to the agent notebook, confirm each source actually returns
something sensible.

In [14]:
# --- Vector search check: policy index ---
policy_store = PineconeVectorStore(index=pc.Index(POLICY_INDEX_NAME), embedding=embeddings)

results = policy_store.similarity_search("Can I work from home?", k=2)

for r in results:
    print("--", r.metadata.get("source_file"))
    print(r.page_content)
    print("+++++++++++++++++++++++++++++++++++\n")


-- Loxford_Company_Policy.pdf
approved in writing by a Vice President or above and documented with People & Culture. 
2. Remote Work Policy 
2.1 Hybrid Work 
Employees may work remotely up to three (3) days per week with their direct manager's approval. The 
remaining two days should be spent at a Loxford office to support in-person collaboration, unless the 
employee's role is designated as fully remote. 
2.2 Fully Remote Arrangements 
Fully remote arrangements require sign-off from a Vice President and a documented home-office setup 
meeting Loxford's minimum standards, including: 
• A dedicated, distraction-free workspace 
• Reliable internet connectivity of at least 25 Mbps download / 10 Mbps upload 
• Ability to join video calls with camera and microphone during core hours 
• Compliance with Loxford's data security and device-encryption requirements 
2.3 Core Hours 
All employees, remote or in-office, are expected to be reachable between 10:00 AM and 3:00 PM in their local
+++++++

In [15]:
# --- Vector search check: product index ---
product_store = PineconeVectorStore(index=pc.Index(PRODUCT_INDEX_NAME), embedding=embeddings)

results = product_store.similarity_search("how do I reset the router", k=2)

for r in results:
    print("--", r.metadata.get("source_file"))
    print(r.page_content)
    print("+++++++++++++++++++++++++++++++++++\n")


-- Loxford_X200_Manual.pdf
apply parental controls. Changes made in the app take effect immediately and do not require a reboot. 
4.2 Guest Network 
To create a guest network, open the app, go to Wi-Fi Settings > Guest Network, and toggle it on. Guest devices 
are isolated from your main network and from each other by default. 
4.3 Firmware Updates 
The X200 checks for firmware updates automatically once per week. To update manually, go to Settings > 
Firmware > Check for Updates in the Companion App. Do not unplug the router during an update — this can 
take up to five minutes and is indicated by a blinking blue Status LED. 
5. Factory Reset 
A factory reset erases all settings, including your Wi-Fi name, password, and any custom configuration, and 
restores the router to its out-of-box state. 
10. Locate the small Reset button on the back panel, next to the WAN port. 
11. Using a paperclip or SIM tool, press and hold the button for 10 seconds. 
12. Release once the Wi-Fi LED begins b

In [16]:
# --- SQL check ---
conn = sqlite3.connect(SQLITE_PATH)

print(conn.execute("SELECT * FROM inventory").fetchall())

conn.close()


[('X200', 'X200 Router', 42, 89.99), ('Y150', 'Y150 Switch', 130, 45.5), ('Z900', 'Z900 Access Point', 17, 129.0)]


---
### Next step
Open **`02_rag_agent_chatbot.ipynb`**. It connects to the two Pinecone indexes and the
SQLite file created above, and wires them into a LangGraph agent that routes each
question to the right source(s) automatically.
